In [14]:
import boto3
import json

endpoint_name = 'jumpstart-dft-hf-tc-bert-base-uncas-20250429-100758'

client = boto3.client('sagemaker-runtime')

text_input = "The moon landing was faked by NASA."

response = client.invoke_endpoint(
    EndpointName=endpoint_name,
    ContentType="text/csv",
    Body=text_input
)

result = response['Body'].read().decode('utf-8')
print(result)


[{"sentence": "The moon landing was faked by NASA.", "probabilities": [0.9975185394287109, 0.0024814882781356573]}]


In [15]:
import pandas as pd
import boto3
import json

client = boto3.client('sagemaker-runtime')
endpoint_name = 'jumpstart-dft-hf-tc-bert-base-uncas-20250429-100758'

df = pd.read_csv('combined_news.csv', on_bad_lines='skip').dropna(subset=['text', 'label'])

df_0 = df[df['label'] == 0].sample(n=500, random_state=42)
df_1 = df[df['label'] == 1].sample(n=500, random_state=42)

df_balanced = pd.concat([df_0, df_1]).sample(frac=1, random_state=42).reset_index(drop=True)

def sanitize_text(text, max_words=250):
    text = str(text).replace('\n', ' ').replace('\r', ' ')
    words = text.split()
    return ' '.join(words[:max_words])

predicted_labels = []
error_count = 0

for i, row in df_balanced.iterrows():
    try:
        text = sanitize_text(row['text'])

        response = client.invoke_endpoint(
            EndpointName=endpoint_name,
            ContentType="text/csv",
            Body=text
        )

        result = json.loads(response['Body'].read().decode())
        probs = result[0]['probabilities']
        predicted_label = int(probs.index(max(probs)))  
        predicted_labels.append(predicted_label)

    except Exception:
        error_count += 1
        predicted_labels.append(None)

df_balanced['predicted_label'] = predicted_labels
df_balanced.to_csv('predicted_fake_news.csv', index=False)

print("Done. Predictions saved to predicted_fake_news.csv")
print(f"Total rows with errors: {error_count}")


Done. Predictions saved to predicted_fake_news.csv
Total rows with errors: 11


In [16]:
import boto3

bucket_name = 'fake-news-vcc'  
s3_file_key = 'predicted_fake_news.csv' 
local_file_path = 'predicted_fake_news.csv'  

s3 = boto3.client('s3')

s3.upload_file(local_file_path, bucket_name, s3_file_key)

print(f"File uploaded to s3://{bucket_name}/{s3_file_key}")


File uploaded to s3://fake-news-vcc/predicted_fake_news.csv
